# Cómputo y concurrencia

Este notebook es la clase de la sección 10.2 completa: cada celda trae su explicación arriba y el código listo.

**Antes de empezar:**

1. Con internet, antes de la clase: `uv sync` dentro de esta carpeta (instala `httpx` y `numpy`).
2. En una terminal aparte, dentro de esta carpeta, arranca el servidor en modo silencioso para que no imprima cada petición y no distorsione los tiempos: `FDD_SILENCIO=1 uv run python servidor.py`. Déjala abierta.
3. Elige como kernel el `.venv` de `10_apis`.

Córrelas **en orden**, una por una (Shift+Enter), y compara con «Deberías ver»: algunas celdas usan lo que definió la anterior. Las celdas con `await` funcionan tal cual porque el notebook ya tiene un event loop corriendo; `asyncio.run(...)` es para archivos `.py`, aquí truena. La página del curso tiene lo mismo, más las tablas y los diagramas: https://rayalucaria.org/fdd_o26/apis/computo-y-concurrencia/

# Dónde pasa el tiempo

Meta: separar el tiempo de una tarea en lo que **ejecuta** y lo que **espera**, medirlo y leer el resultado en un Gantt.

Pregunta de la página: **¿por qué 20 peticiones tardan 20 veces lo que una?** Y la que sigue: si tardan eso, ¿qué parte del tiempo es tu programa trabajando?

Meta: separar el tiempo de una tarea en lo que **ejecuta** y lo que **espera**, medirlo y leer el resultado en un Gantt.

| Término | Qué es | Lo ves con |
|---|---|---|
| **tarea** | Una unidad de trabajo con un inicio y un fin, por ejemplo una llamada a la API | la celda 1.1 |
| **tramo que ejecuta** | El tiempo en que la CPU corre instrucciones de la tarea | la columna `CPU` de la celda 1.1 |
| **tramo que espera** | El tiempo en que la tarea no avanza porque aguarda algo de fuera (la red, un disco, un timer) | la columna `reloj` de la celda 1.1 |
| **CPU-bound** | Tarea cuyo tiempo es casi todo ejecución: más CPU la acelera | el bucle de la celda 1.1 |
| **I/O-bound** | Tarea cuyo tiempo es casi todo espera: más CPU no la acelera | la llamada de la celda 1.1 |
| **secuencial** | Ejecutar las tareas una tras otra: la siguiente empieza cuando la anterior terminó | la celda 1.2 |

### En corto

- **Una tarea = ejecuta + espera.** El tiempo total (el *reloj*) es la suma de los dos tramos.
- **Una llamada a una API casi no ejecuta: espera.** Tu CPU está disponible durante esa espera y no hace nada.
- **En secuencial las esperas se suman**: 20 llamadas de 0.2 s son 4 s, no 0.2 s.

### Ejecuta y espera

**Definición.** Una **tarea** es una unidad de trabajo con inicio y fin. Su duración se parte en dos tramos que no se traslapan: el que **ejecuta** (la CPU corre instrucciones) y el que **espera** (la tarea no puede avanzar hasta que algo de fuera responda). Hay un tercer estado que sólo aparece cuando varias tareas comparten una CPU: **lista**, la tarea ya podría ejecutar pero otra tiene la CPU y le toca esperar su turno. Los Gantt de las páginas siguientes lo dibujan aparte; la diferencia entre el reloj y el tiempo de CPU de abajo incluye también ese turno.

Una llamada a una API tiene los dos: ejecutar es armar la petición y leer el JSON; esperar es el viaje por la red y el tiempo del servidor (la latencia de que-es-una-api).

**Para qué sirve separarlos.** Cada tramo se acelera de forma distinta. Ejecutar se acelera con más CPU o menos instrucciones; esperar no se acelera con nada de tu lado, sólo se **solapa** con otra cosa. Si no sabes cuál tramo domina, optimizas el equivocado.

**Si lo ignoras.** Es el error más caro de la sección: comprar más núcleos o reescribir en otro lenguaje un programa que pasa el 99 % del tiempo esperando una respuesta.

### CPU-bound e I/O-bound

**Definición.** Una tarea es **CPU-bound** si casi todo su tiempo es ejecución; es **I/O-bound** (entrada/salida: red, disco) si casi todo su tiempo es espera.

La diferencia se ve comparando dos relojes: el **reloj** (`time.perf_counter`, lo que pasó en la pared) y el **tiempo de CPU** (`time.process_time`, lo que la CPU trabajó para tu proceso). Si coinciden, la tarea ejecutó todo el rato; si el reloj es mucho mayor, esperó.

**Haz (celda 1.1):** mide una llamada con `?lento=0.2` y un bucle puro de Python con los dos relojes.

## 1.1 · Dos relojes: una llamada y un bucle

In [ ]:
import time

import requests

from cable import URL


def cuenta(n=3_000_000):
    suma = 0
    for i in range(n):
        suma += i * i
    return suma


def mide(nombre, tarea):
    reloj0, cpu0 = time.perf_counter(), time.process_time()
    tarea()
    reloj = time.perf_counter() - reloj0
    cpu = time.process_time() - cpu0
    print(f"{nombre:22} reloj {reloj:5.2f} s   CPU {cpu:5.2f} s   CPU/reloj {cpu / reloj:5.0%}")


mide("llamada ?lento=0.2", lambda: requests.get(f"{URL}/ventas?lento=0.2", timeout=5))
mide("bucle puro de Python", cuenta)

**Qué hace cada línea:**

- `from cable import URL` — la dirección del servidor, de 10.1.
- `cuenta(n)` — suma los cuadrados de `0` a `n - 1` con un bucle de Python: no toca la red ni el disco, sólo calcula.
- `mide(nombre, tarea)` — corre `tarea` y lee los dos relojes antes y después.
- `time.perf_counter()` — el reloj de pared: segundos reales transcurridos.
- `time.process_time()` — los segundos que la CPU trabajó para este proceso; no cuenta el tiempo dormido o esperando.
- `cpu / reloj` — la fracción del tiempo en que la tarea ejecutó.

**Deberías ver:**

```text
llamada ?lento=0.2     reloj  0.20 s   CPU  0.00 s   CPU/reloj    1%
bucle puro de Python   reloj  0.35 s   CPU  0.35 s   CPU/reloj  100%
```

- En la llamada, la CPU trabaja una fracción mínima del reloj: el resto la tarea **esperó**. Es I/O-bound.
- En el bucle, el reloj y la CPU son casi iguales: nunca esperó. Es CPU-bound.
- La misma máquina, el mismo Python, dos tareas de naturaleza opuesta. Por eso la herramienta correcta cambia con la tarea (páginas 2 a 5).

> `process_time` cuenta el proceso completo del notebook, no sólo la tarea; con el kernel en reposo la diferencia es de milisegundos. Los números son de la máquina del profesor, y los tuyos serán otros: lo que se conserva es la proporción.

### Veinte llamadas en secuencia

**Definición.** Un programa es **secuencial** cuando ejecuta las tareas una tras otra: la tarea *k + 1* no empieza hasta que la *k* terminó.

Lo que pasa con 20 llamadas a `?lento=0.2` se lee en un Gantt: una barra por tarea sobre el eje del tiempo, con el tramo que ejecuta y el que espera, y debajo una fila con lo que hace la CPU.

*Figura en la página del curso: «Tareas en secuencia: las esperas se suman y la CPU queda inactiva».*

En el Gantt de arriba hay tres cosas que leer:

- Cada tarea empieza hasta que termina la anterior: el tiempo total es la **suma** de las duraciones.
- Los tramos que esperan son largos y los que ejecutan son astillas.
- La fila de la CPU está vacía durante toda espera: la máquina **podía** hacer otra cosa y no hizo nada.

**Haz (celda 1.2):** llama 20 veces seguidas y reparte el tiempo entre ejecutar y esperar.

## 1.2 · Veinte llamadas en secuencia

In [ ]:
import time

import requests

from cable import URL

N = 20
reloj0, cpu0 = time.perf_counter(), time.process_time()
for _ in range(N):
    requests.get(f"{URL}/ventas?lento=0.2", timeout=5)
reloj = time.perf_counter() - reloj0
cpu = time.process_time() - cpu0

print(f"{N} llamadas en serie: {reloj:.2f} s  ({reloj / N:.3f} s por llamada)")
print(f"ejecuta (CPU):  {cpu:.2f} s  ({cpu / reloj:.1%})")
print(f"espera:         {reloj - cpu:.2f} s  ({(reloj - cpu) / reloj:.1%})")
print(f"sueño pedido al servidor: {N * 0.2:.1f} s")

**Qué hace cada línea:**

- `for _ in range(N)` — 20 vueltas; cada `requests.get` bloquea hasta recibir la respuesta, así que la siguiente no sale antes.
- `reloj - cpu` — lo que no fue CPU de tu proceso: espera.
- `N * 0.2` — lo que pidió el programa: 20 pausas de 0.2 s. El reloj no puede ser menor a eso.

**Deberías ver:**

```text
20 llamadas en serie: 4.04 s  (0.202 s por llamada)
ejecuta (CPU):  0.03 s  (0.7%)
espera:         4.01 s  (99.3%)
sueño pedido al servidor: 4.0 s
```

El total fue **4.04 s**. De ese tiempo, **99.3 %** fue espera; la CPU de tu proceso ejecutó el resto.

### Por qué ese porcentaje decide todo

La regla que ordena el resto de la sección: **el tiempo de una tarea I/O-bound es casi todo espera, y la espera se puede solapar con otra espera**. Si 20 llamadas esperan al mismo tiempo en lugar de una tras otra, el total se acerca a lo que tarda **una**, no a la suma.

| Dónde está el tiempo | Qué lo acelera | Qué no lo acelera | Páginas |
|---|---|---|---|
| **Espera** (I/O-bound) | Solapar esperas: hilos o `asyncio` | Más núcleos; un algoritmo más rápido | 2, 3 |
| **Ejecución** en Python (CPU-bound) | Repartir el cálculo en procesos | Hilos (por el GIL, ver el-gil) | 2 |
| **Ejecución** sobre muchos datos numéricos | Vectorizar: sacar el bucle de Python | Más hilos | 4 |

Los 20 y los 0.2 s son parámetros de esta demostración. Lo que no depende de ellos es la pregunta que haces antes de tocar el código: **¿cuánto de mi tiempo es espera?** Si la respuesta es 95 %, el máximo que puedes ganar ejecutando más rápido es 5 %.

> Con 100 o 500 llamadas la secuencial no se mide en esta clase: 100 llamadas son 20 s y 500 son 100 s. Son N × 0.2 s, **calculado, no medido**. Es suficiente: la suma de esperas es exacta porque no hay nada que se solape.

### Al diseñar / al revisar código de IA, busca…

- **Un `for` que llama a una API, a una base de datos o a un archivo en cada vuelta**: si las llamadas no dependen una de otra, las esperas se están sumando sin necesidad.
- **Optimización de la parte equivocada**: reescrituras de algoritmo, caché de cálculos o más CPU sobre un programa que pasa casi todo su reloj esperando. Pide la medición de `reloj` contra `CPU` antes de aceptar el cambio.
- **Tiempos sin medir**: «esto es rápido» o «esto es lento» sin `perf_counter`, o con un solo número en lugar de la proporción ejecuta/espera.
- **Una llamada de red sin `timeout`**: la espera no tiene tope (ver metodos-y-status).
- **Mezclar los dos tipos de tarea en el mismo bucle** (llamar y luego calcular con la respuesta): cada tramo necesita su propia herramienta, y se separan antes de elegir una.

# Concurrente, paralelo y asíncrono

Meta: saber cuál de las tres palabras describe lo que hace tu programa, y qué se rompe cuando varios hilos tocan el mismo dato.

Meta: saber cuál de las tres palabras describe lo que hace tu programa, y qué se rompe cuando varios hilos tocan el mismo dato.

Veinte llamadas de 0.2 s tardan 4 s en serie. Alguien te dice «hazlo concurrente», otro «paralelo», otro «asíncrono». ¿Son tres nombres de lo mismo? No: **cada palabra afirma algo distinto, y se comprueba con un reloj.**

| Término | Qué es | Lo ves con |
|---|---|---|
| **concurrente** | Dos tareas son concurrentes si sus ciclos de vida (de empezar a terminar) se **solapan** en el tiempo | la celda 2.1 |
| **paralelo** | Dos cosas **se ejecutan en el mismo instante**, en dos CPU lógicas distintas | la celda 2.3 |
| **asíncrono** | La llamada **regresa antes de tener el resultado**, y quien espera **cede el control** de forma explícita | la celda 2.2 |
| **condición de carrera** | El resultado depende del orden en que se intercalan dos hilos que tocan el mismo dato | la celda 2.4 |
| **`Lock`** | Un candado: sólo un hilo a la vez puede estar dentro del bloque que protege | la celda 2.5 |

### En corto

- **Concurrente** habla de *tiempo*: ciclos de vida que se solapan. No dice cuántas CPU hay.
- **Paralelo** habla de *CPU*: ejecución simultánea, así que necesita al menos 2.
- **Asíncrono** es un *modelo de programación* (cómo escribes la espera), no una celda de la matriz.
- **Con GIL, hilos y `asyncio` son concurrentes y no paralelos.** Los procesos son ambas cosas.

### Qué se ejecuta: proceso, hilo, CPU lógica, GIL

La definición larga de estos cuatro está en el-gil; aquí sólo la tabla que usas para leer el resto de la página.

| | Qué es | Memoria | Lo que obtienes de él |
|---|---|---|---|
| **proceso** | Un programa corriendo | propia: no comparte variables con otro proceso | su propio intérprete y su propio GIL |
| **hilo** | Una línea de ejecución dentro de un proceso | **compartida** con los demás hilos del proceso | solapar esperas; barato de crear |
| **CPU lógica** | Lo que el sistema puede poner a ejecutar algo a la vez | no aplica | el tope de ejecuciones simultáneas |
| **GIL** | El candado que deja a un solo hilo ejecutar Python por intérprete | no aplica | hilos que se turnan para ejecutar |

Lo que cambia todo lo que sigue: **los hilos comparten memoria y los procesos no.** Lo primero abarata la comunicación y abre la condición de carrera; lo segundo la aísla y obliga a mandar los datos de ida y vuelta.

### Concurrente: ciclos de vida que se solapan

**Definición:** dos tareas son concurrentes si una **empieza antes de que la otra termine**. No dice que se ejecuten a la vez, sólo que ninguna espera a que acabe la otra para empezar.

**Haz (celda 2.1):** las mismas 20 llamadas de la página anterior, ahora con 32 hilos.

## 2.1 · Las mismas 20 llamadas con 32 hilos

In [ ]:
import time
import requests
from concurrent.futures import ThreadPoolExecutor
from cable import URL

def pide(i):
    return requests.get(f"{URL}/ventas?lento=0.2", timeout=5).status_code

inicio = time.perf_counter()
with ThreadPoolExecutor(max_workers=32) as hilos:
    codigos = list(hilos.map(pide, range(20)))
print(f"{len(codigos)} llamadas, status {set(codigos)}, {time.perf_counter() - inicio:.2f} s")

**Qué hace cada línea:**

- `pide(i)` — una llamada `GET` con `?lento=0.2`: el servidor duerme 0.2 s y contesta. Devuelve sólo el status.
- `ThreadPoolExecutor(max_workers=32)` — un grupo de hasta 32 hilos del mismo proceso.
- `hilos.map(pide, range(20))` — lanza las 20 llamadas sin esperar a que termine la anterior y junta los resultados en orden.
- `set(codigos)` — los status distintos que hubo; si todo salió bien, uno solo.

**Deberías ver:** en la máquina del profesor (tu número cambia con tu máquina y su carga),

```text
20 llamadas, status {200}, 0.23 s
```

- **0.23 s contra 4.04 s en serie.** Las 20 llamadas empezaron antes de que terminara la primera: sus ciclos de vida se solaparon. Eso es concurrencia.
- No hubo ejecución simultánea de Python: el tiempo que se ahorró es **tiempo de espera** solapado, no cálculo repartido.

*Figura en la página del curso: «Las mismas tareas con hilos: esperas solapadas, ejecución turnada».*

En el Gantt de arriba, **las barras de espera se montan unas sobre otras y las de ejecución no.** Eso es lo que hace el GIL: mientras un hilo espera a la red suelta el candado y otro ejecuta; dos hilos nunca ejecutan Python a la vez. La medición que lo prueba para código que calcula (4 hilos ≈ 1 hilo) es la celda 3.2 de el-gil.

### Asíncrono: la llamada regresa antes del resultado

**Definición:** una llamada es asíncrona si **regresa de inmediato con una tarea pendiente (`Task`) del resultado** y, para usarlo, quien espera lo pide con `await`, **cediendo el control** a otra tarea mientras tanto. Es una forma de escribir el programa; no dice cuántas CPU usa.

**Haz (celda 2.2):** una tarea que tarda 0.2 s, creada y consultada antes de que termine.

## 2.2 · Una tarea que todavía no termina

In [ ]:
import asyncio

async def cuenta():
    await asyncio.sleep(0.2)
    return 42

tarea = asyncio.create_task(cuenta())
print("justo después de crearla, terminó:", tarea.done())
resultado = await tarea
print("después de await, terminó:", tarea.done(), "y devolvió", resultado)

**Qué hace cada línea:**

- `async def cuenta()` — define una **corrutina**: una función que puede pausarse en un `await`.
- `await asyncio.sleep(0.2)` — espera 0.2 s *cediendo el control* para que otra tarea ejecute.
- `asyncio.create_task(cuenta())` — programa la corrutina y **regresa al instante**, sin esperar el resultado.
- `tarea.done()` — dice si ya terminó.
- `await tarea` — aquí sí cedes el control hasta tener el resultado. En un notebook se usa `await` directo en la celda; `asyncio.run(...)` sólo va en archivos `.py`.

**Deberías ver:**

```text
justo después de crearla, terminó: False
después de await, terminó: True y devolvió 42
```

- La primera línea es `False`: **la llamada regresó antes de tener el resultado.** Esa es la parte asíncrona.
- Lo que hace falta para solapar muchas llamadas (el *event loop*, `gather`) es la siguiente página: asyncio-y-el-event-loop.

### Paralelo: ejecución en el mismo instante

**Definición:** dos cosas son paralelas si **se ejecutan en el mismo instante**. Eso exige al menos 2 CPU lógicas, y en Python con GIL exige **procesos** (o la compilación *free-threaded* de 3.14, que ya vive en el-gil): cada uno trae su intérprete y su GIL.

La función que calcula vive en `trabajo_computo.py`, un archivo importable: un proceso nuevo vuelve a importarla, y una celda no vive en ningún archivo (el porqué está en el-gil).

**Haz (celda 2.3):** el mismo cálculo, 4 veces, con 1, 2 y 4 procesos.

## 2.3 · El mismo cálculo con 1, 2 y 4 procesos

In [ ]:
import os
import time
from concurrent.futures import ProcessPoolExecutor
from trabajo_computo import calcula

def mide(n_procesos):
    inicio = time.perf_counter()
    with ProcessPoolExecutor(max_workers=n_procesos) as procesos:
        list(procesos.map(calcula, [5_000_000] * 4))
    return time.perf_counter() - inicio

print("CPU lógicas:", os.cpu_count())
for n in (1, 2, 4):
    print(f"{n} proceso(s): {mide(n):.2f} s")

**Qué hace cada línea:**

- `from trabajo_computo import calcula` — la función que suma en Python puro; vive en el archivo, no en la celda.
- `mide(n_procesos)` — corre las 4 llamadas de `calcula` repartidas entre `n_procesos` procesos y devuelve cuánto tardó.
- `[5_000_000] * 4` — cuatro trabajos idénticos.
- `os.cpu_count()` — cuántas CPU lógicas ve el sistema: el tope de la ejecución simultánea.

**Deberías ver:**

```text
CPU lógicas: 8
1 proceso(s): 2.34 s
2 proceso(s): 1.45 s
4 proceso(s): 1.00 s
```

- **2.34 s con 1 proceso, 1.45 s con 2, 1.00 s con 4**, en una máquina con 8 CPU lógicas. Con 2 procesos hay dos ejecuciones en el mismo instante.
- Pasar de 1 a 2 acorta el total porque **hay ejecución simultánea**. Con más procesos que CPU libres se turnan y el total deja de bajar; aquí no se llegó a ese límite (4 procesos sobre 8 CPU). El total de 4 procesos (1.00 s) queda por encima del ideal (2.34 / 4 = 0.58 s) porque **arrancar un proceso y copiarle los datos cuesta tiempo**.

*Figura en la página del curso: «Con dos CPU, dos tareas ejecutan en el mismo instante».*

El Gantt de arriba dibuja tareas con espera, para mostrar el mecanismo; la celda 2.3 es puro cálculo. En él, **las barras de ejecución se montan**: eso no pasaba con hilos. Cada barra corre sobre una CPU distinta; las filas de CPU lo muestran. Crear un proceso cuesta más que crear un hilo y los datos viajan copiados, así que para trabajo corto no compensa.

### La matriz: concurrente × paralelo

Las dos primeras palabras forman una matriz de 2 × 2. La tercera no entra: asíncrono dice *cómo escribes* la espera, no *cuándo* ejecuta.

*Figura en la página del curso: «Concurrente por paralelo: dónde cae cada estrategia».*

| Estrategia | Concurrente | Paralelo | Qué consigue en Python con GIL |
|---|:---:|:---:|---|
| secuencial | no | no | nada: una tarea tras otra |
| hilos | **sí** | no | solapa esperas |
| `asyncio` | **sí** | no | solapa esperas, con un solo hilo |
| procesos | **sí** | **sí** | solapa esperas **y** reparte cálculo |

- **Sí, hilos y `asyncio` son concurrentes y no paralelos**: la celda 2.1 ahorró tiempo sin que dos hilos ejecutaran Python a la vez.
- **Paralelo sin concurrente sí existe**: *una* sola tarea repartida entre varias CPU, como una operación de NumPy sobre un arreglo grande dentro de una librería compilada (lo ves en vectorizar). Hay ejecución simultánea y no hay varias tareas en curso.

> «Más rápido» tiene dos causas distintas. Si el trabajo **espera**, basta con solapar (concurrente). Si **calcula**, sólo ayuda ejecutar a la vez (paralelo). Con la pregunta de la página anterior, ¿espera o calcula?, eliges la columna.

### Cuando hilos comparten un dato: la condición de carrera

Los hilos comparten memoria (primera tabla). Eso es lo que los hace baratos y lo que los hace peligrosos: **`contador += 1` no es una operación indivisible.** Son tres pasos: leer el valor, sumar 1, escribir el resultado. Si otro hilo lee entre el primer y el tercer paso, una de las dos sumas se pierde.

**Haz (celda 2.4):** 4 hilos suman 1 al mismo contador, 100 000 veces cada uno, sin candado.

## 2.4 · Condición de carrera: contador sin candado

In [ ]:
import sys
import threading

sys.setswitchinterval(1e-6)   # fuerza que los hilos se turnen más seguido

contador = 0

def suma_uno(x):
    return x + 1

def suma(veces):
    global contador
    for _ in range(veces):
        actual = contador              # 1. leer
        contador = suma_uno(actual)    # 2-3. sumar y escribir

hilos = [threading.Thread(target=suma, args=(100_000,)) for _ in range(4)]
for h in hilos:
    h.start()
for h in hilos:
    h.join()
print(f"esperado {4 * 100_000:,}, obtenido {contador:,}")
sys.setswitchinterval(0.005)   # lo regresa al valor por defecto

**Qué hace cada línea:**

- `sys.setswitchinterval(1e-6)` — baja a 1 µs el intervalo en que Python deja que otro hilo ejecute (por defecto 5 ms). Sin esto la carrera ocurre sólo *a veces*, y eso es lo que la hace difícil de depurar.
- `global contador` — todos los hilos escriben en la misma variable.
- `actual = contador` y `contador = suma_uno(actual)` — la lectura y la escritura separadas por una llamada a función. Desde Python 3.10 el intérprete sólo cambia de hilo al entrar a una función o al volver al inicio de un bucle; la llamada abre el hueco donde otro hilo se cuela. Sin ella, la carrera existe igual pero casi nunca se ve.
- `h.start()` / `h.join()` — arranca cada hilo y espera a que todos terminen.
- `sys.setswitchinterval(0.005)` — devuelve el intervalo a 5 ms para que no afecte a las celdas siguientes.

**Deberías ver:**

```text
esperado 400,000, obtenido 190,207
```

- **Esperado 400,000, obtenido 190,207.** Faltan sumas: dos hilos leyeron el mismo valor y escribieron el mismo resultado.
- El programa **no truena ni avisa**. Da un número equivocado y distinto en cada corrida.

El arreglo es un **`Lock`**: sólo un hilo a la vez entra al bloque que lo usa, y los otros esperan su turno.

**Haz (celda 2.5):** el mismo conteo, con candado.

## 2.5 · El mismo conteo con candado

In [ ]:
import sys
import threading

sys.setswitchinterval(1e-6)   # el mismo forzado de la celda 2.4

contador = 0
candado = threading.Lock()

def suma_uno(x):
    return x + 1

def suma_con_candado(veces):
    global contador
    for _ in range(veces):
        with candado:
            actual = contador
            contador = suma_uno(actual)

hilos = [threading.Thread(target=suma_con_candado, args=(100_000,)) for _ in range(4)]
for h in hilos:
    h.start()
for h in hilos:
    h.join()
print(f"esperado {4 * 100_000:,}, obtenido {contador:,}")
sys.setswitchinterval(0.005)

**Qué hace cada línea:**

- `threading.Lock()` — crea el candado.
- `with candado:` — entra al bloque sólo si nadie más lo tiene; al salir lo suelta, incluso si hay un error.
- Dentro del bloque están juntas la lectura y la escritura: ningún otro hilo se cuela en medio.

**Deberías ver:**

```text
esperado 400,000, obtenido 400,000
```

- **Esperado y obtenido coinciden.** El precio: mientras un hilo está dentro del `with`, los otros esperan, así que el trozo protegido se ejecuta en serie. Protege lo mínimo.

> Con procesos no hay carrera sobre una variable, porque no comparten memoria; pero tampoco ven los cambios del otro. Cada proceso trabaja sobre su copia y el resultado se devuelve y se junta al final.

### Al diseñar / al revisar código de IA, busca…

- **«Concurrente» o «paralelo» sin medir.** Pregunta cuántas CPU usa y qué tarda con 1 y con N; una respuesta con hilos que calculan no puede ser «paralelo».
- **Hilos o `asyncio` para acelerar un cálculo en Python puro.** Son concurrentes, no paralelos: no ejecutan a la vez.
- **Un contador, lista o diccionario compartido entre hilos, escrito sin `Lock`** (`+=`, `append` encadenado con lectura previa). Es una carrera aunque hoy el resultado salga bien.
- **Un `Lock` que envuelve todo el cuerpo de la función.** Convierte los hilos en una ejecución en serie con más pasos.
- **Procesos con una función definida en la celda o un `lambda`.** El proceso nuevo no la encuentra; debe vivir en un archivo importable.

# asyncio y el event loop

Meta: esperar 100 o 500 respuestas con un solo hilo, y saber qué línea de código rompe esa ganancia.

Meta: esperar 100 o 500 respuestas con un solo hilo, y saber qué línea de código rompe esa ganancia.

La pregunta: **¿cómo espera 100 peticiones un solo hilo?** Los hilos ya solapan esperas (ver concurrente-paralelo-asincrono), pero cada hilo cuesta memoria y el sistema operativo decide cuándo cambiarlo. `asyncio` hace lo mismo con un hilo y con cambios que decide tu código.

| Término | Definición de una línea | Lo ves en |
|---|---|---|
| **corrutina** | Función definida con `async def`; al llamarla no corre, devuelve un objeto que alguien tiene que esperar | celda 3.1 |
| **`await`** | Pausa esta corrutina hasta que termine lo esperado y deja que el loop corra otras | celda 3.1 |
| **tarea** | Corrutina ya programada en el loop, que avanza sin que la esperes | celda 3.2 |
| **event loop** | El bucle de un hilo que guarda las corrutinas pendientes y, cada vez que una hace `await`, corre otra que ya puede avanzar | celda 3.2 |

### Una corrutina no corre al llamarla

**Haz (celda 3.1):** llama una corrutina sin `await` y luego con `await`.

## 3.1 · Una corrutina sin await y con await

In [ ]:
import asyncio

async def saluda():
    await asyncio.sleep(0.1)
    return "listo"

c = saluda()
print(type(c).__name__)
print(await c)

**Qué hace cada línea:**

- `async def saluda()` — define una corrutina.
- `await asyncio.sleep(0.1)` — espera 0.1 s cediendo el hilo al loop.
- `c = saluda()` — no ejecuta nada: guarda la llamada en un objeto `coroutine`.
- `await c` — ahora sí corre y devuelve `"listo"`. El `await` de nivel superior funciona porque el notebook ya tiene un loop corriendo.

**Deberías ver:**

```text
coroutine
listo
```

> En un notebook usa `await` suelto, como en esta celda. `asyncio.run(...)` sólo va en un archivo `.py`: dentro de un notebook truena con `RuntimeError: asyncio.run() cannot be called from a running event loop`, porque el loop ya existe.

### El event loop, con una traza de tiempos

**Haz (celda 3.2):** tres tareas que esperan 1 s cada una; la traza imprime cuándo empieza y termina cada una.

## 3.2 · Tres esperas que se solapan

In [ ]:
import asyncio
import time

t0 = time.perf_counter()

def marca(texto):
    print(f"{time.perf_counter() - t0:4.2f} s  {texto}")

async def trabajo(nombre):
    marca(f"{nombre} empieza")
    await asyncio.sleep(1)
    marca(f"{nombre} termina")

await asyncio.gather(trabajo("A"), trabajo("B"), trabajo("C"))
marca("las tres terminaron")

**Qué hace cada línea:**

- `marca(texto)` — imprime los segundos desde `t0` y un texto.
- `await asyncio.sleep(1)` — la corrutina se pausa; el loop pasa a la siguiente.
- `asyncio.gather(...)` — programa las tres corrutinas como tareas y espera a todas.

**Deberías ver:**

```text
0.00 s  A empieza
0.00 s  B empieza
0.00 s  C empieza
1.00 s  A termina
1.00 s  B termina
1.00 s  C termina
1.00 s  las tres terminaron
```

- Las tres empiezan casi en el mismo instante y terminan casi juntas: **3 esperas de 1 s tardaron 1 s, no 3 s.**
- Todo ocurrió en un hilo. El loop corre A hasta su `await`, luego B, luego C, y vuelve a cada una cuando su espera acaba.

*Figura en la página del curso: «El event loop reparte un hilo entre corrutinas».*

### Cien peticiones con un hilo: `gather` y `Semaphore`

`requests` bloquea el hilo mientras espera, así que no sirve dentro del loop. El cliente para `asyncio` es `httpx.AsyncClient`.

**Haz (celda 3.3):** 20, 100 y 500 llamadas a `/ventas?lento=0.2`, con 32 en vuelo como máximo.

## 3.3 · 20, 100 y 500 llamadas con asyncio

In [ ]:
import asyncio
import time
import httpx
from cable import URL

async def pide(cliente, sem):
    async with sem:
        r = await cliente.get(f"{URL}/ventas?lento=0.2")
        return r.status_code

async def lote(n):
    sem = asyncio.Semaphore(32)
    limites = httpx.Limits(max_connections=32)
    async with httpx.AsyncClient(limits=limites, timeout=30) as cliente:
        return await asyncio.gather(*(pide(cliente, sem) for _ in range(n)))

for n in (20, 100, 500):
    inicio = time.perf_counter()
    codigos = await lote(n)
    print(f"{n:>3} llamadas, asyncio con 32: {time.perf_counter() - inicio:5.2f} s  ({set(codigos)})")

**Qué hace cada línea:**

- `asyncio.Semaphore(32)` — deja pasar como máximo 32 corrutinas a la vez por el bloque `async with sem`; las demás esperan su turno.
- `httpx.Limits(max_connections=32)` — el mismo tope en el cliente: 32 conexiones abiertas.
- `await cliente.get(...)` — la corrutina cede el hilo mientras el servidor duerme 0.2 s.
- `asyncio.gather(*(...))` — programa las `n` corrutinas y devuelve sus resultados en orden.

**Deberías ver:**

```text
 20 llamadas, asyncio con 32:  0.24 s  ({200})
100 llamadas, asyncio con 32:  0.92 s  ({200})
500 llamadas, asyncio con 32:  3.41 s  ({200})
```

- El tiempo crece por tandas: cada tanda de 32 llamadas cuesta ~0.2 s, porque ése es el tope y no el hilo.

*Figura en la página del curso: «Mismas tareas, un hilo, gather».*

### Hilos y `asyncio` con el mismo límite

Una comparación es honesta si los dos modelos tienen el mismo tope: aquí, 32 llamadas en vuelo. El secuencial sólo se midió con 20 llamadas; 100 y 500 son el cálculo `N × 0.2 s`.

| Llamadas | Secuencial | Hilos (`max_workers=32`) | `asyncio` (`Semaphore(32)`) |
|---:|---:|---:|---:|
| 20 | 4.04 s | 0.23 s | 0.24 s |
| 100 | 20 s (calculado, no medido) | 0.82 s | 0.92 s |
| 500 | 100 s (calculado, no medido) | 3.32 s | 3.41 s |

Tiempos de la máquina del profesor; los tuyos cambian, la forma no. Con el mismo tope, hilos y `asyncio` tardan lo mismo: **el límite lo pone la concurrencia permitida, no el modelo.**

Subir el tope es un parámetro. Con 500 llamadas y 100 en vuelo, los hilos tardan 1.52 s; `asyncio` con 100 conexiones de `httpx` tardó 10.91 s, **más lento que con 32** (una sola corrida, no repetida, sin celda en el notebook). El servidor no es la causa: el cliente gastó 10.9 s de CPU. El costo está en el **pool de conexiones** de `httpx` (las conexiones abiertas que el cliente guarda y reutiliza entre peticiones), no en el modelo; por eso se mide antes de subir un límite.

- Para **esperar**, 100 hilos o 1000 corrutinas es válido; la regla "no más workers que CPU" es de trabajo que calcula.
- La diferencia práctica: una corrutina ocupa unos KB y cambia sin el sistema operativo; un hilo ocupa más y lo cambia el sistema. Con miles de esperas, eso decide.
- `asyncio.to_thread(requests.get, ...)` **no es `asyncio` real**: son hilos: corre en el **executor** por omisión del loop (un pool de `min(32, CPUs + 4)` hilos que ejecuta funciones fuera del loop). Sirve para meter una librería que bloquea, pero no tiene las ventajas de la tabla.

### El contraejemplo: `await` uno tras otro

`async` no vuelve concurrente nada por sí mismo. Un `await` dentro de un `for` espera cada llamada antes de lanzar la siguiente.

**Haz (celda 3.4):** las mismas 20 llamadas, cada una con su `await` en serie.

## 3.4 · await en serie

In [ ]:
import time
import httpx
from cable import URL

inicio = time.perf_counter()
async with httpx.AsyncClient(timeout=30) as cliente:
    for _ in range(20):
        r = await cliente.get(f"{URL}/ventas?lento=0.2")
print(f"20 llamadas con await en serie: {time.perf_counter() - inicio:.2f} s")

**Qué hace cada línea:**

- `for _ in range(20)` — recorre 20 veces.
- `await cliente.get(...)` — el loop podría correr otras corrutinas durante la espera, pero **no hay otras**: la siguiente vuelta no empieza hasta que ésta termina.

**Deberías ver:**

```text
20 llamadas con await en serie: 4.05 s
```

- Es el tiempo del secuencial (20 × 0.2 s), con sintaxis `async`. La corrutina cedió el hilo y nadie lo usó.
- El arreglo es `gather` (celda 3.3): programar todas antes de esperar.

*Figura en la página del curso: «await en serie: async sin concurrencia».*

### `time.sleep` contra `asyncio.sleep`

`asyncio.sleep` es una espera que cede el hilo; `time.sleep` es una espera que **se queda con él**. El loop no puede correr nada mientras una corrutina llama `time.sleep`.

**Haz (celda 3.5):** la traza de la celda 3.2, con `time.sleep` en lugar de `asyncio.sleep`.

## 3.5 · time.sleep dentro de una corrutina

In [ ]:
import asyncio
import time

t0 = time.perf_counter()

def marca(texto):
    print(f"{time.perf_counter() - t0:4.2f} s  {texto}")

async def trabajo(nombre):
    marca(f"{nombre} empieza")
    time.sleep(1)
    marca(f"{nombre} termina")

await asyncio.gather(trabajo("A"), trabajo("B"), trabajo("C"))
marca("las tres terminaron")

**Qué hace cada línea:**

- `time.sleep(1)` — duerme el hilo entero, y con él el loop. A no cede nada: termina antes de que B empiece.

**Deberías ver:**

```text
0.00 s  A empieza
1.00 s  A termina
1.00 s  B empieza
2.00 s  B termina
2.00 s  C empieza
3.00 s  C termina
3.00 s  las tres terminaron
```

- Tres esperas de 1 s tardan 3 s: la traza es secuencial.
- Cualquier llamada que bloquea (`time.sleep`, `requests.get`, leer un archivo grande, un cálculo) tiene el mismo efecto.

### Un cálculo dentro del loop lo bloquea

Un `await` es el único punto donde el loop recupera el control. Un cálculo largo no tiene `await`: mientras corre, **ninguna otra corrutina avanza**, ni siquiera un temporizador.

**Haz (celda 3.6):** un latido imprime cada 0.1 s; en medio, una corrutina calcula sin `await`.

## 3.6 · Un cálculo que no cede el event loop

In [ ]:
import asyncio
import time
from trabajo_computo import calcula

async def latido(veces):
    for i in range(veces):
        print(f"{time.perf_counter() - t0:4.2f} s  latido {i}")
        await asyncio.sleep(0.1)

async def calculo():
    await asyncio.sleep(0.15)
    inicio = time.perf_counter()
    calcula()
    print(f"      el cálculo duró {time.perf_counter() - inicio:.2f} s sin ceder el hilo")

t0 = time.perf_counter()
await asyncio.gather(latido(5), calculo())

**Qué hace cada línea:**

- `latido(veces)` — corrutina que imprime la hora y espera 0.1 s, `veces` veces.
- `calcula()` — la función de cálculo de `trabajo_computo.py`; es código normal, sin `await`.
- `asyncio.gather(latido(5), calculo())` — las dos corrutinas en el mismo loop.

**Deberías ver:**

```text
0.00 s  latido 0
0.10 s  latido 1
      el cálculo duró 2.28 s sin ceder el hilo
2.43 s  latido 2
2.53 s  latido 3
2.63 s  latido 4
```

- Los latidos que debían salir cada 0.1 s se detienen durante el cálculo: el loop estuvo bloqueado 2.28 s.
- No es lo que pasa con el GIL: el GIL le quita el turno a un hilo cada ~5 ms (`sys.getswitchinterval()`, ver el-gil). El loop es cooperativo: nadie interrumpe a una corrutina, y sólo recupera el control cuando ella llega a un `await`.

*Figura en la página del curso: «Un cálculo sin await detiene el loop».*

### La salida: sacar el cálculo del loop

Hay dos caminos, según qué calcule el trabajo:

| El trabajo | Dónde correrlo | Con qué |
|---|---|---|
| espera en una librería que bloquea | un hilo | `await asyncio.to_thread(f, ...)` |
| calcula en Python | otro proceso (tiene su propio GIL) | `await loop.run_in_executor(pool, f, ...)` con un `ProcessPoolExecutor` |

`run_in_executor` manda la función a otro hilo o proceso y devuelve un objeto que sí se puede esperar con `await`: el loop sigue corriendo mientras tanto. La función vive en `trabajo_computo.py` porque un proceso nuevo la importa.

**Haz (celda 3.7):** el mismo latido y el mismo cálculo, con el cálculo en un proceso.

## 3.7 · El cálculo en un proceso

In [ ]:
import asyncio
import time
from concurrent.futures import ProcessPoolExecutor
from trabajo_computo import calcula

async def latido(veces):
    for i in range(veces):
        print(f"{time.perf_counter() - t0:4.2f} s  latido {i}")
        await asyncio.sleep(0.1)

async def calculo(pool):
    await asyncio.sleep(0.15)
    loop = asyncio.get_running_loop()
    inicio = time.perf_counter()
    await loop.run_in_executor(pool, calcula)
    print(f"      el cálculo duró {time.perf_counter() - inicio:.2f} s en otro proceso")

t0 = time.perf_counter()
with ProcessPoolExecutor(max_workers=1) as pool:
    await asyncio.gather(latido(5), calculo(pool))

**Qué hace cada línea:**

- `asyncio.get_running_loop()` — el loop que corre esta celda.
- `loop.run_in_executor(pool, calcula)` — corre `calcula` en el proceso del pool; el `await` cede el hilo mientras tanto.
- `ProcessPoolExecutor(max_workers=1)` — un solo proceso, suficiente para una función.

**Deberías ver:**

```text
0.00 s  latido 0
0.10 s  latido 1
0.20 s  latido 2
0.30 s  latido 3
0.40 s  latido 4
      el cálculo duró 2.39 s en otro proceso
```

- Los latidos siguen saliendo cada 0.1 s mientras el cálculo corre en otro proceso; el loop no se enteró.
- 2.39 s contra 2.28 s de la celda 3.6: la diferencia es arrancar el proceso, que el pool crea en el primer `run_in_executor` y queda dentro del cronómetro.

**Al diseñar / al revisar código de IA, busca:**

- `async def` con `await` dentro de un `for` sobre llamadas independientes: es secuencial con otra sintaxis; pide `asyncio.gather`.
- `time.sleep(`, `requests.get(` o `open(...).read()` dentro de una `async def`: bloquea el loop y no lo parece.
- `gather` sin `Semaphore` ni `Limits` sobre miles de llamadas: abre miles de conexiones a la vez y el servidor responde `429` o cierra.
- Un bucle de cálculo dentro de una `async def`: sácalo con `run_in_executor` y un proceso.
- `asyncio.run(` dentro de una celda de notebook, o `asyncio.to_thread(requests.get, ...)` presentado como `asyncio`: lo segundo son hilos.

# Vectorizar: cuando el trabajo calcula sobre muchos datos

Meta: saber qué hacer cuando el tiempo no se va en esperar sino en calcular sobre muchos datos, medir cuánto gana una operación sobre un arreglo y calcular el techo de cualquier mejora que reparta trabajo.

Meta: saber qué hacer cuando el tiempo no se va en esperar sino en calcular sobre muchos datos, medir cuánto gana una operación sobre un arreglo y calcular el techo de cualquier mejora que reparta trabajo.

Las páginas anteriores repartían **esperas**: hilos y `asyncio` ganan porque mientras una petición espera, otra avanza. Un cálculo sobre cinco millones de números no espera nada. Un bucle de Python lo recorre uno por uno, y repartirlo en hilos no lo acelera por el GIL (el-gil). ¿Qué cambia si en lugar de repartir el bucle lo eliminas?

| Término | Qué es | Lo ves con |
|---|---|---|
| **vectorización** | Aplicar una operación a un arreglo completo en una sola llamada, en vez de un elemento por vez desde un bucle de Python | la celda 4.2 |
| **arreglo contiguo** | Una secuencia de valores del mismo tipo guardados uno junto a otro en memoria | la celda 4.1 |
| **SIMD** | Instrucción de CPU que opera sobre varios valores a la vez en un solo paso | la figura de abajo |
| **speedup** | Tiempo de la versión base entre tiempo de la versión mejorada | la celda 4.2 |
| **ley de Amdahl** | El techo del speedup cuando sólo una parte del trabajo se puede repartir | la celda 4.4 |

### En corto

- **Vectorizar** no reparte el trabajo entre CPU: lo mueve de un bucle del intérprete a un bucle en C que corre sobre memoria contigua.
- **La ganancia viene sobre todo de sacar el bucle de Python**; SIMD es un factor secundario.
- **Sólo sirve si la operación es la misma para todos los elementos**. Lógica distinta por elemento, objetos arbitrarios o datos que no caben en memoria quedan fuera.

### Qué guarda cada uno

Una lista de Python guarda **referencias a objetos**: cada `float` es un objeto aparte con su cabecera, y para sumarlos el intérprete sigue cada referencia, revisa el tipo y despacha la operación. Un arreglo de NumPy guarda **los valores crudos, todos del mismo tipo, en un bloque contiguo**.

**Haz (celda 4.1):** crea un arreglo de cinco millones de `float64` y mira qué ocupa; mide aparte lo que cuesta crearlo.

## 4.1 · Un arreglo de float64

In [ ]:
import time
import numpy as np

n = 5_000_000

t0 = time.perf_counter()
a = np.arange(n, dtype=np.float64)
crear_s = time.perf_counter() - t0

print("dtype:", a.dtype, "| bytes por elemento:", a.itemsize)
print("bytes totales:", a.nbytes)
print("contiguo:", a.flags["C_CONTIGUOUS"])
print(f"crear el arreglo: {crear_s:.4f} s")

**Qué hace cada línea:**

- `np.arange(n, dtype=np.float64)` crea los números de 0 a `n - 1` ya como decimales de 64 bits.
- `itemsize` y `nbytes` muestran que cada elemento ocupa 8 bytes, sin cabecera de objeto.
- `C_CONTIGUOUS` confirma que los valores están uno junto a otro.
- La creación se mide **aparte**: no se cuenta dentro de la comparación de la siguiente celda.

**Deberías ver:**

```text
dtype: float64 | bytes por elemento: 8
bytes totales: 40000000
contiguo: True
crear el arreglo: 0.0065 s
```

### Bucle de Python contra una sola llamada

La tarea: la suma de cuadrados de los `n` números. Las dos versiones calculan lo mismo; cambia quién recorre los datos.

**Haz (celda 4.2):** mide el bucle de Python sobre una lista y la operación vectorizada sobre el arreglo, y verifica que coinciden.

## 4.2 · Bucle contra operación vectorizada

In [ ]:
import time
import numpy as np

n = 5_000_000
a = np.arange(n, dtype=np.float64)
lista = a.tolist()  # la conversión queda fuera de la medición

def suma_cuadrados_bucle(xs):
    total = 0.0
    for x in xs:
        total += x * x
    return total

def suma_cuadrados_numpy(arr):
    return float(np.sum(arr * arr))

suma_cuadrados_numpy(a)  # corrida de calentamiento, se descarta

t0 = time.perf_counter()
r_bucle = suma_cuadrados_bucle(lista)
bucle_s = time.perf_counter() - t0

t0 = time.perf_counter()
r_numpy = suma_cuadrados_numpy(a)
numpy_s = time.perf_counter() - t0

print(f"bucle de Python: {bucle_s:.3f} s")
print(f"NumPy:           {numpy_s:.4f} s")
print(f"speedup:         {bucle_s / numpy_s:.0f}x")
print("coinciden:", np.isclose(r_bucle, r_numpy, rtol=1e-9))

**Qué hace cada línea:**

- `lista = a.tolist()` prepara la entrada del bucle fuera del cronómetro, para no cobrarle la conversión.
- `suma_cuadrados_bucle` es el bucle que ya conoces: una multiplicación y una suma por elemento, ejecutadas por el intérprete.
- `arr * arr` y `np.sum` son **dos operaciones sobre el arreglo completo**; el recorrido ocurre en C.
- La llamada sin guardar el resultado es el calentamiento: la primera corrida se descarta.
- `np.isclose` compara con tolerancia relativa porque las dos versiones suman en distinto orden y los decimales no son asociativos.

**Deberías ver:**

```text
bucle de Python: 0.188 s
NumPy:           0.0119 s
speedup:         16x
coinciden: True
```

Son tiempos de la máquina del profesor; los tuyos cambian, y la proporción también puede cambiar.

*Figura en la página del curso: «Un bucle elemento a elemento contra una operación sobre el arreglo completo».*

### Qué cambió, y cuánto le toca a cada cosa

| Qué cambió | Efecto |
|---|---|
| **Tipos homogéneos** | No se revisa el tipo de cada elemento: se sabe una vez para todo el arreglo |
| **Memoria contigua** | La CPU lee los datos en orden y la caché los tiene a mano |
| **El bucle sale del intérprete** | Desaparecen el despacho por elemento, la referencia a objetos y el conteo de referencias |
| **SIMD** | Una instrucción procesa varios valores a la vez; ayuda, pero es un factor **secundario** |

**La ganancia no se debe principalmente a SIMD.** Casi toda sale de quitar el costo por elemento del intérprete. SIMD sólo es posible porque los datos son homogéneos y contiguos, pero lo que lo prueba es la comparación de arriba, no el nombre de la instrucción.

### `float64`, no `int64`

Un entero de NumPy tiene 64 bits y **no crece**: si el resultado no cabe, da la vuelta sin avisar. Un `int` de Python no tiene tope. La suma de cuadrados de 0 a 5 millones pasa de `9.2e18`, el máximo de `int64`.

**Haz (celda 4.3):** suma los mismos cuadrados con `int64` y compara con el valor exacto.

## 4.3 · int64 se desborda

In [ ]:
import numpy as np

n = 5_000_000
enteros = np.arange(n, dtype=np.int64)

con_int64 = int(np.sum(enteros * enteros))
exacto = (n - 1) * n * (2 * n - 1) // 6  # fórmula cerrada, enteros de Python
con_float64 = float(np.sum(np.arange(n, dtype=np.float64) ** 2))

print("máximo de int64:", np.iinfo(np.int64).max)
print("exacto (Python):", exacto)
print("int64 (NumPy):  ", con_int64)
print("float64 (NumPy):", f"{con_float64:.6e}")

**Qué hace cada línea:**

- `np.iinfo(np.int64).max` es el mayor entero que cabe en 64 bits con signo.
- `exacto` usa la fórmula de la suma de cuadrados con enteros de Python, que no se desbordan.
- `con_int64` suma en `int64`: el resultado pasa el máximo y sale **otro número, sin error**.
- `con_float64` pierde precisión en los últimos dígitos, pero conserva la magnitud.

**Deberías ver:**

```text
máximo de int64: 9223372036854775807
exacto (Python): 41666654166667500000
int64 (NumPy):   4773166019248396768
float64 (NumPy): 4.166665e+19
```

### Cuándo vectorizar no sirve

| Caso | Por qué | Qué hacer |
|---|---|---|
| **Lógica distinta por elemento** (ramas, estado que cambia al avanzar) | No hay una sola operación que aplicar al arreglo | Procesos, o reescribir con máscaras (`np.where`) si la lógica lo permite |
| **Objetos de Python** (cadenas, diccionarios, instancias) | El arreglo de objetos guarda referencias otra vez: sin tipos crudos, sin ganancia | Procesos, o convertir los datos a un `dtype` numérico si lo admiten |
| **Datos que no caben en memoria** | `arr * arr` crea otro arreglo del mismo tamaño | Procesar por bloques, o una herramienta lazy como Polars (9.3) |
| **Pocos elementos** | El costo fijo de cada llamada pesa más que el bucle | Un bucle normal |
| **Cada elemento depende del anterior** | No se pueden calcular a la vez | Reformular, o aceptar el bucle |

**NumPy libera el GIL** durante las operaciones grandes sobre arreglos: mientras C recorre los datos, otro hilo de Python puede correr. Es un dato útil, no un sustituto de medir; el mecanismo del GIL está en el-gil.

### Amdahl: el techo de repartir el trabajo

> Si falta tiempo, la sección de Amdahl se omite.

Si repartes un trabajo entre `P` unidades de cómputo, sólo la parte repartible se acelera. **La ley de Amdahl** dice cuánto: con `S` la fracción del tiempo que **no** se puede repartir, el speedup con `P` unidades es:

`speedup(P) = 1 / (S + (1 - S) / P)`

Con `P` tendiendo a infinito el techo es `1 / S`. Si la mitad del trabajo es serial, nunca pasas de 2x, tengas las unidades que tengas.

| `S` (fracción serial) | `P` = 2 | `P` = 4 | `P` = 8 | Techo (`P` infinito) |
|---|---:|---:|---:|---:|
| 0.10 | 1.82 | 3.08 | 4.71 | 10.0 |
| 0.25 | 1.60 | 2.29 | 2.91 | 4.0 |
| 0.50 | 1.33 | 1.60 | 1.78 | 2.0 |

**Haz (celda 4.4):** calcula la tabla y mira cuánto rinde duplicar `P` cuando `S = 0.25`.

## 4.4 · Ley de Amdahl

In [ ]:
def amdahl(S, P):
    return 1 / (S + (1 - S) / P)

for S in (0.10, 0.25, 0.50):
    fila = [f"{amdahl(S, P):.2f}" for P in (2, 4, 8)]
    print(f"S={S:.2f}", fila, "techo:", round(1 / S, 1))

print("S=0.25, de 4 a 8 unidades:", round(amdahl(0.25, 8) / amdahl(0.25, 4), 2), "veces más")

**Qué hace cada línea:**

- `amdahl(S, P)` es la fórmula tal cual.
- El bucle imprime una fila por cada `S` con `P` = 2, 4 y 8, más el techo `1 / S`.
- La última línea compara 8 unidades contra 4: duplicar `P` no duplica el speedup.

**Deberías ver:**

```text
S=0.10 ['1.82', '3.08', '4.71'] techo: 10.0
S=0.25 ['1.60', '2.29', '2.91'] techo: 4.0
S=0.50 ['1.33', '1.60', '1.78'] techo: 2.0
S=0.25, de 4 a 8 unidades: 1.27 veces más
```

*Figura en la página del curso: «Speedup teórico de Amdahl según la fracción serial».*

**Sólo limita de verdad al trabajo que calcula.** Al repartir esperas también hay parte serial (la CPU de cada tarea se turna bajo el GIL, como dibuja el Gantt de hilos), pero es pequeña: en la celda 1.2 fue 0.7 % del tiempo. Con `S` tan chico Amdahl casi no limita, y 100 hilos o 1000 corrutinas esperando a la vez son válidos mientras el servidor y el sistema aguanten. Y el `S` real incluye lo que no ves en el código: arrancar procesos, serializar datos de ida y vuelta, juntar resultados. Por eso el speedup medido queda por debajo de la curva.

### Al diseñar / al revisar código de IA, busca…

- **Un `for` de Python sobre un arreglo o una columna** para sumar, multiplicar o filtrar: casi siempre es una operación vectorizable.
- **Hilos para acelerar un cálculo numérico**: el GIL lo impide; vectoriza o usa procesos.
- **Acumuladores en `int64` o `int32` sobre sumas grandes**: desbordan sin aviso. Revisa el `dtype`.
- **Una medición que incluye crear el arreglo o convertir la lista**, o que no descarta la primera corrida: infla o desinfla el speedup.
- **Una promesa de «N veces más rápido con N procesos»** sin hablar de la fracción serial ni del costo de serializar.

# Elegir la estrategia

Pregunta: dado un trabajo concreto, ¿qué uso, y cómo me doy cuenta de que elegí mal?

| Término | Qué es | Lo ves con |
|---|---|---|
| **árbol de decisión** | Las preguntas, en orden, que llevan de «dónde pasa el tiempo» a una estrategia | la figura de abajo |
| **anti-patrón** | Código que funciona y aun así anula la ventaja de la estrategia que dice usar | las celdas 5.1 y 5.2 |
| **pool** | Un grupo de hilos o procesos que se crea una vez y se reutiliza | la celda 5.2 |

### En corto

- **Primero mide dónde pasa el tiempo** (donde-pasa-el-tiempo); la estrategia es consecuencia, no preferencia.
- **Esperar** se resuelve con hilos o `asyncio`; **calcular en Python**, con procesos; **calcular sobre arreglos**, vectorizando.
- **Un anti-patrón no falla: tarda lo mismo que el secuencial** (o más). Por eso se busca con un comando, no con la vista.

### El árbol

*Figura en la página del curso: «Árbol de decisión: de dónde pasa el tiempo a la estrategia».*

| Si el trabajo... | Estrategia | Por qué | Página |
|---|---|---|---|
| **espera** (red, disco, base de datos) y son pocas esperas, con código bloqueante | hilos | Mientras un hilo espera, el GIL se libera y otro avanza | concurrente-paralelo-asincrono |
| **espera** y son muchas, con librerías `async` | `asyncio` | Una corrutina que espera cuesta mucho menos que un hilo | asyncio-y-el-event-loop |
| **calcula en Python** (un bucle que no cabe en un arreglo) | procesos | Cada proceso tiene su intérprete y su GIL: es el único paralelismo para código Python puro con GIL | concurrente-paralelo-asincrono |
| **calcula sobre arreglos** numéricos | vectorizar | Saca el bucle del intérprete; no hay nada que repartir | vectorizar |
| **mezcla** las dos cosas | combinar | Espera con `asyncio` o hilos y manda el cálculo a un pool de procesos | esta página |

### Los costos, medidos

Mismo servidor, mismo límite de concurrencia declarado (32), una sola corrida en esta máquina con `8` CPU.

| Trabajo | Estrategia | Tiempo |
|---|---|---:|
| 20 llamadas de 0.2 s | secuencial | `4.04` s |
| 20 llamadas de 0.2 s | 32 hilos | `0.23` s |
| 20 llamadas de 0.2 s | `asyncio`, semáforo de 32 | `0.24` s |
| 500 llamadas de 0.2 s | 32 hilos | `3.32` s |
| 500 llamadas de 0.2 s | `asyncio`, semáforo de 32 | `3.41` s |
| 4 cálculos en Python (`calcula(5_000_000)`) | 1 proceso | `2.34` s |
| 4 cálculos en Python (`calcula(5_000_000)`) | 4 procesos | `1.00` s |
| suma de cuadrados | bucle de Python | `0.188` s |
| suma de cuadrados | NumPy | `0.0119` s (`16` veces más rápido) |

- **Para esperar, hilos y `asyncio` quedan parejos** con el mismo límite de 32: elegir entre ellos es cuestión de librerías y de cuántas esperas, no de velocidad. Con el tope en 100 dejaron de estarlo (1.52 s contra 10.91 s en una sola corrida, ver asyncio-y-el-event-loop): un límite nuevo se vuelve a medir.
- **Los procesos ganan en cálculo, no en espera**: lanzarlos para llamadas de red añade arranque y copia de datos sin ganar nada.
- **Repartir tiene techo**: 4 procesos dieron 2.3 veces, no 4 (Amdahl, vectorizar). Los 16x de NumPy son sobre otra carga (5 M `float64`, no los 20 M enteros de `calcula()`), así que esas dos cifras no se comparan entre sí.

### Anti-patrones, con su comando

| Anti-patrón | Qué pasa | Cómo lo delatas |
|---|---|---|
| **`await` en serie**: `for ...: await ...` | Cada espera empieza cuando termina la anterior: es secuencial con otra sintaxis | El tiempo total es `4.05` s, no `0.24` s; revisa el Gantt de asyncio-y-el-event-loop |
| **Bloquear el loop**: `time.sleep` o `requests.get` dentro de `async def` | Nada más avanza mientras tanto: un cálculo de `2.28` s detuvo el loop en la celda 3.6 | El latido de la celda 3.6 se detiene |
| **Pool por petición**: crear el pool dentro del bucle | Paga el arranque de hilos o procesos cada vez | La celda 5.2 |
| **`lambda` en procesos** | El proceso hijo recibe la función serializada con `pickle`, y una `lambda` no se puede serializar | La celda 5.1 |
| **Más workers que CPU, en trabajo que calcula** | Los procesos extra se turnan por los mismos núcleos y no suman: 16 tareas cortas tardaron `1.31` s con 8 procesos y `1.32` s con 16, en `8` CPU (una corrida, fuera del notebook) | Compara `max_workers` con `os.cpu_count()` |

La última fila **sólo aplica a trabajo que calcula**. Para esperar, 100 hilos o mil corrutinas es válido: no ocupan CPU mientras esperan.

**Haz (celda 5.1):** manda una `lambda` a un proceso.

## 5.1 · Una lambda a un proceso

In [ ]:
from concurrent.futures import ProcessPoolExecutor

with ProcessPoolExecutor(max_workers=2) as pool:
    futuro = pool.submit(lambda x: x * 2, 21)
    try:
        print(futuro.result())
    except Exception as e:
        print(type(e).__name__)

**Qué hace cada línea:**

- `ProcessPoolExecutor(max_workers=2)` — dos procesos hijo.
- `pool.submit(lambda ...)` — serializa la función con `pickle` para enviarla al hijo.
- `futuro.result()` — devuelve el resultado o relanza el error que ocurrió en el camino.

**Deberías ver:**

```text
PicklingError
```

- El mensaje de `e` cambia entre versiones de Python (3.13 y 3.14 lo redactan distinto); el tipo es el mismo.

- **La función que va a un proceso debe vivir en un módulo importable** (como `trabajo_computo.py`) o ser una función incorporada: así `pickle` la nombra y el hijo la encuentra.

**Haz (celda 5.2):** compara un pool por llamada con un pool para todas.

## 5.2 · Un pool por llamada contra un pool para todas

In [ ]:
import time
from concurrent.futures import ProcessPoolExecutor

t = time.perf_counter()
for i in range(10):
    with ProcessPoolExecutor(max_workers=2) as pool:
        pool.submit(abs, -i).result()
print(f"un pool por llamada : {time.perf_counter() - t:.2f} s")

t = time.perf_counter()
with ProcessPoolExecutor(max_workers=2) as pool:
    for i in range(10):
        pool.submit(abs, -i).result()
print(f"un pool para todas  : {time.perf_counter() - t:.2f} s")

**Qué hace cada línea:**

- `pool.submit(abs, -i)` — trabajo mínimo, para que el tiempo sea el del pool y no el del cálculo.
- Primer bloque — crea y destruye un pool en cada vuelta.
- Segundo bloque — crea el pool una vez y lo reutiliza.

**Deberías ver:**

```text
un pool por llamada : 0.73 s
un pool para todas  : 0.07 s
```

- La diferencia es el costo de arrancar procesos, y se paga **por petición** si el pool vive dentro del manejador.

### Qué decide una API frente a esto

Un servidor web recibe muchas peticiones a la vez y tiene que escoger la misma estrategia que tú:

- **Un hilo por conexión.** El servidor de la clase (`ThreadingHTTPServer`) lanza un hilo por cada conexión: simple, y válido porque el trabajo de cada petición casi siempre espera.
- **Un event loop.** Servidores `async` atienden miles de conexiones en un hilo; sirven mientras ningún manejador bloquee el loop.
- **Varios procesos.** Los servidores de producción de Python arrancan varios procesos, cada uno con su GIL, para usar todos los núcleos.

Un manejador que **calcula mucho** no se arregla con más hilos: se mueve el cálculo a un pool de procesos, a otro servicio o a una cola (sección 10.4).

### Cierre


### Al diseñar, y al revisar código de IA, busca...

- **`for ...: await ...` sobre una lista de llamadas.** Si no hay `gather` ni tareas, es secuencial.
- **`requests`, `time.sleep` o lectura de archivos dentro de `async def`.** Bloquea el loop de todos.
- **Un `ThreadPoolExecutor` o `ProcessPoolExecutor` creado dentro de un bucle o de un manejador.** Sácalo y reutilízalo.
- **Una `lambda` o función anidada enviada a procesos.** Moverla a un módulo.
- **`max_workers` fijado en un número grande para trabajo que calcula.** Compáralo con `os.cpu_count()`.
- **Hilos para sumar o multiplicar números en un bucle.** El GIL los turna; vectoriza o usa procesos.

La siguiente clase es **patrones y protocolos de comunicación**: qué hacer cuando dos programas no esperan una respuesta inmediata, sino un evento.